# Descriptor database from ORCA outputs

Spec-driven pipeline. Everything that depends on the molecule family lives in one YAML file under `specs/`:
which atoms are labelled and how they are found (`numbering`), which descriptors are computed on which labels
(`descriptors.orca`, `descriptors.descripytor`), and how conformers are aggregated and filtered.

Stages (each also available as `python -m desc_utils.run <stage> --spec ...`):

1. **number** – build the atom numbering table (one row per conformer, `label -> 1-based index`)
2. **preprocess** – ORCA output -> pseudo-Gaussian log -> feather, needed by DescriPyTor (unchanged legacy step)
3. **extract** – per-conformer descriptors from both engines
4. **aggregate** – Boltzmann statistics per molecule (`_min`, `_max`, `_lowE`, `_boltz`) and noise filtering

In [ ]:
import os
import sys
import subprocess
from datetime import datetime
from pathlib import Path

import pandas as pd

ROOT_DIR = Path.cwd().resolve()          # run this notebook from the repository root
sys.path.insert(0, str(ROOT_DIR))

from desc_utils.spec import load_spec
from desc_utils.numbering import build_numbering_table, read_numbering_table, write_numbering_table
from desc_utils.extract import run_extraction
from desc_utils.extract.descripytor import to_schema_b
from desc_utils.aggregate import aggregate_conformers, apply_filter

pd.set_option("display.max_columns", 60)
print(f"Repository root: {ROOT_DIR}")

### Choose the spec

`specs/lnih2_pyox.yaml` is the LNiH2 / PyOx library; `specs/test_molecules_at1_at2.yaml` labels the first two atoms of small test molecules.
To study a new molecule family, copy one of these files and edit `atoms`, `numbering` and `descriptors`; validate it with
`python -m desc_utils.run validate --spec specs/<file>.yaml`.

In [ ]:
SPEC_PATH = ROOT_DIR / "specs" / "lnih2_pyox.yaml"      # or "test_molecules_at1_at2.yaml"

spec = load_spec(SPEC_PATH)
print(f"spec: {spec.name}\nlabels: {spec.labels}\nnumbering backend: {spec.numbering['backend']}\ndata root: {spec.root}")

# 1. Atom numbering

Set `REBUILD_NUMBERING = True` to run the spec's numbering backend over every conformer folder. Otherwise the existing
CSV (`spec.output['numbering_csv']`) is reused, which lets you inspect and hand-correct indices before extraction.
Rows whose `numbering_status` is not `ok` are listed and skipped downstream.

In [ ]:
REBUILD_NUMBERING = False
numbering_csv = ROOT_DIR / spec.output["numbering_csv"]

if REBUILD_NUMBERING or not numbering_csv.exists():
    df_num_full = build_numbering_table(spec, repo_root=ROOT_DIR)
    write_numbering_table(df_num_full, numbering_csv)
    bad = df_num_full[df_num_full["numbering_status"] != "ok"]
    print(f"wrote {numbering_csv}: {len(df_num_full)} conformers, {len(bad)} not numbered cleanly")
    if len(bad):
        display(bad[["path", "numbering_status", "numbering_notes"]])

df_num = read_numbering_table(numbering_csv, spec)
df_num.head()

# 2. Preprocessing for DescriPyTor (legacy step)

DescriPyTor reads Gaussian-style logs. `orca2gaussian/orca_2_gauss.py` writes a pseudo-Gaussian `.log` into every
conformer folder from the ORCA outputs, and `logs_to_feather` turns each log into the `.feather` file `Molecules` loads.
Both steps skip folders that already have their output unless `OVERWRITE_FEATHER = True`.
This step is slated to be replaced by a direct ORCA -> DescriPyTor-JSON converter (`descriptors.descripytor.input.kind: json`).

In [ ]:
OVERWRITE_FEATHER = False
orca_folder = ROOT_DIR / spec.root

script_path = ROOT_DIR / "orca2gaussian" / "orca_2_gauss.py"
try:
    subprocess.run([sys.executable, str(script_path), "--orca-folder", str(orca_folder)],
                   capture_output=True, text=True, check=True)
except subprocess.CalledProcessError as exc:
    print(f"orca2gaussian failed for {orca_folder}: {exc.stderr.strip() or exc}")

DESCRIPYTOR_DIR = ROOT_DIR / "DescriPyTor"
if str(DESCRIPYTOR_DIR) not in sys.path:
    sys.path.insert(0, str(DESCRIPYTOR_DIR))
from M2_data_extractor.feather_extractor import logs_to_feather   # noqa: E402  (git submodule update --init if missing)

n_done = n_skipped = 0
for _, row in df_num.iterrows():
    conf_dir = ROOT_DIR / row["path"]
    logs = sorted(conf_dir.glob("*.log"))
    if not logs:
        print(f"no .log file in {conf_dir}")
        continue
    if any(conf_dir.glob("*.feather")) and not OVERWRITE_FEATHER:
        n_skipped += 1
        continue
    try:
        logs_to_feather(str(conf_dir), feather_dir=str(conf_dir))
        n_done += 1
    except Exception as exc:
        print(f"feather conversion failed in {conf_dir}: {exc}")
os.chdir(ROOT_DIR)
print(f"feather files written: {n_done}, already present: {n_skipped}")

# 3. Per-conformer descriptors

`run_extraction` resolves every labelled term of the spec to atom indices for each conformer and runs the ORCA-side
families (`xyz` = morfeus geometry, `charges`, `nbo`, `redox`) and the DescriPyTor families. Use `families=[...]` to
run a subset, e.g. `["descripytor"]`.

In [ ]:
df_desc = run_extraction(spec, df_num, repo_root=ROOT_DIR)

per_conformer_csv = ROOT_DIR / spec.output["per_conformer_csv"]
df_desc.to_csv(per_conformer_csv, index=False)
print(f"wrote {per_conformer_csv}: {df_desc.shape[0]} conformers x {df_desc.shape[1]} columns")
df_desc.head()

# 4. Conformer aggregation and filtering

Boltzmann weights come from `aggregate.energy_column` at `aggregate.temperature_K`. `apply_filter` runs
`filter_numeric_descriptors` with the options of the spec's `filter` section and re-attaches the metadata columns.

In [ ]:
df_agg = aggregate_conformers(df_desc, spec)
print(f"aggregated: {df_agg.shape[0]} molecules x {df_agg.shape[1]} columns")

df_final, report = apply_filter(df_agg, spec)
print("Initial numeric descriptors:", report["initial_numeric_count"])
for key in ["removed_low_occ_nbo_columns", "removed_rydberg", "removed_ni_4f_rydberg", "removed_nmr_aniso",
            "removed_phi_theta", "removed_zero_std", "removed_low_ratio", "removed_high_corr"]:
    if key in report:
        print(f"  {key}: {len(report[key])}")
print("Final numeric descriptors:", report["final_numeric_count"])

output_csv = ROOT_DIR / spec.output["aggregated_csv"].format(date=datetime.now().strftime("%Y%m%d"), name=spec.name)
df_final.to_csv(output_csv, index=False)
print(f"wrote {output_csv}")
df_final.head()

### Interoperability with DescriPyTor's own tooling

For any conformer, the labelled DescriPyTor section of the spec can be resolved to the index-based JSON
("Schema B") that DescriPyTor's GUI and `descriptor_extractor.py` consume.

In [ ]:
to_schema_b(spec.descripytor, df_num.iloc[0])